# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [ ]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI
from pydantic import BaseModel, Field, ValidationError
# Make sure your OPENAI_API_KEY is set in the environment
from dotenv import load_dotenv
import os
import ollama

load_dotenv(r"C:\Users\prahn\OneDrive\Documents\IITM-Pravartak\Pravartak_Practice\practice_scripts\.env")
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [4]:
#Note: This does not return the id of the job. It must be extracted from the snippets list
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    zero_shot_prompt = f"""Based only on the job description snippet provided, extract and return the following information as JSON.
1. company — the company doing the hiring
2. role — the job title
3. years_experience_required — the minimum experience required (integer). If experience required is not specified, return null.

Snippet:
{snippet_text}
"""
    return [{'role': 'user', 'content': zero_shot_prompt}]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    few_shot_prompt = f"""
Using the three example job descriptions and their outputs as reference,
extract the following information from the job description provided:

- company
- role
- years_experience_required

Return the result as JSON. When experience is specified, years_experience_required
must be an integer. If experience required is not specified, return null

Example 1:
Job description:
ABC is recruiting a Senior Engineer to join their team. The ideal candidate
must have 10+ years of experience.

Output:
{{"company": "ABC", "role": "Senior Engineer", "years_experience_required": 10}}

Example 2:
Job description:
DEF Inc is recruiting an HR Manager to join their management. Applicants
must have three to five years of experience.

Output:
{{"company": "DEF Inc", "role": "HR Manager", "years_experience_required": 3}}

Example 3:
Job description:
GHI is recruiting an Analyst to join their data team. Freshers can also
apply if they can demonstrate experience.

Output:
{{"company": "GHI", "role": "Analyst", "years_experience_required": null}}

Now extract the information from this job description:

{snippet_text}
"""
    return [{'role': 'user', 'content': few_shot_prompt}]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    structured_prompt = """
ROLE: You are an expert recruiter.
CONTEXT: The user needs specific details extracted from a job description provided as natural language text.
TASK: 
1. Analyze the provided job description
2. Extract the three fields:
 - "company": the company doing the hiring
 - "role": the job title
 - "years_experience_required": the minimum experience required (integer)
3. If experience required is not specified, return null
FORMAT: Return ONLY a valid JSON object using exactly this schema:
{
  "company": string,
  "role": string,
  "years_experience_required": integer or null
}
"""
    return [{'role': 'system', 'content': structured_prompt}, {'role': 'user', 'content': snippet_text}]    


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    cot_prompt = f"""Based only on the job description snippet provided, extract:
1. company — the company doing the hiring
2. role — the job title
3. years_experience_required — the minimum experience required (integer). 
Think step by step about the information in the job description before providing your final JSON answer.
When experience is specified, years_experience_required must be an integer. If experience required is not specified, return null.
Return the final answer as JSON.

Snippet:
{snippet_text}
"""
    return [{'role': 'user', 'content': cot_prompt}]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [25]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """

    text = text.strip()

    # Try plain JSON first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Handle prose followed by ```json ... ```
    if "```json" in text:
        text = text.split("```json", 1)[1]
        text = text.split("```", 1)[0].strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return None

    return None


is_local=False # this is a flag to determine whether to use the local ollama client or the OpenAI API client. Set to True if using ollama, False for OpenAI API.

def compute_cost_usd(
    rates: dict,
    prompt_tokens: int,
    completion_tokens: int,
    model="gpt-4o-mini"
) -> float:

    model_rates = rates.get(model)

    if model_rates is None:
        return 0.0

    return round(
        prompt_tokens * model_rates["in"]
        + completion_tokens * model_rates["out"],
        6
    )


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    snippet_ID = snippet['id']
    messages = STRATEGIES[strategy_name](snippet['snippet'])
    output = {}
    if is_local:
        ollama_client = ollama.AsyncClient()
        started = time.time()
        response = await ollama_client.chat( model='gemma3:4b',
        messages=messages,
        options={
        'temperature': 0.0},  
        )
        elapsed = round(time.time() - started, 3)
        raw_response = response['message']['content']
        
        output['strategy_name'] = strategy_name
        output['snippet_ID'] = snippet_ID
        output['raw_response'] = raw_response
        output['parsed_extraction'] = parse_response(raw_response)
        output['cost_USD'] = compute_cost_usd(RATES, prompt_tokens=response.prompt_eval_count, completion_tokens=response.eval_count)
        output['latency_s'] = elapsed
        
        return output
    else:
       
        started = time.time()
    
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.0,
        )

        elapsed = round(time.time() - started, 3)

        raw_response = response.choices[0].message.content

        output['strategy'] = strategy_name
        output['snippet_id'] = snippet_ID
        output['raw_response'] = raw_response
        output['parsed_extraction'] = parse_response(raw_response)

        output['cost_usd'] = compute_cost_usd(
            RATES,
            prompt_tokens=response.usage.prompt_tokens,
            completion_tokens=response.usage.completion_tokens
        )

        output['latency_s'] = elapsed

        return output


async def run_all(strategies, snippets) -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = []
    for key in strategies.keys():
        for i in range (len(snippets)):
            tasks.append(run_one(key, snippets[i]))
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
# Run it
results = await run_all(STRATEGIES, snippets)
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'parsed_extraction': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'cost_usd': 3.6e-05,
 'latency_s': 1.88}

In [28]:
with open("run_all_MP1.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
def score_accuracy(llm_op: dict | None, golden: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    
    if llm_op is None:
        return 0

    score = 0
    for field in ['company', 'role', 'years_experience_required']:
        extracted_value = llm_op.get(field)
        gold_value = golden.get(field)

        # Normalize string values for comparison
        if isinstance(extracted_value, str):
            extracted_value = extracted_value.strip().lower()
        if isinstance(gold_value, str):
            gold_value = gold_value.strip().lower()

        if extracted_value == gold_value:
            score += 1
    
    return score

results_test = results.copy()
for i in range(len(results)):
    snippet_id = results[i]['snippet_id']
    golden_entry = golden.get(snippet_id)
    if golden_entry is not None:
        accuracy_score = score_accuracy(results[i]['parsed_extraction'], golden_entry)
        results[i]['accuracy'] = accuracy_score
    else:
        results[i]['accuracy'] = None  # or some default value if no golden entry is found

# **parse_success** — did the response parse cleanly?
for ele in results:
    #print(ele['parsed_extraction'])
    if isinstance(ele['parsed_extraction'], dict):
        ele['parse_success'] = 1
    else:
        ele['parse_success'] = 0

#test_df = pd.DataFrame(results)
#test_df[['strategy', 'snippet_id', 'parsed_extraction', 'accuracy','parse_success']].sort_values(by=['snippet_id']).to_csv('mp1_results_scored.csv', index=False)



In [35]:
results[37]

{'strategy': 'cot',
 'snippet_id': 'j08',
 'raw_response': 'To extract the required information from the job description snippet, let\'s analyze it step by step:\n\n1. **Company**: The company doing the hiring is mentioned at the beginning of the snippet. It is "Wonka Confectionery Ltd."\n  \n2. **Role**: The job title is specified as "Senior UX Researcher."\n\n3. **Years of Experience Required**: The snippet states "Five years of UX research experience or equivalent portfolio required." This indicates that the minimum experience required is 5 years.\n\nNow, let\'s compile this information into the requested JSON format:\n\n```json\n{\n  "company": "Wonka Confectionery Ltd.",\n  "role": "Senior UX Researcher",\n  "years_experience_required": 5\n}\n```',
 'parsed_extraction': {'company': 'Wonka Confectionery Ltd.',
  'role': 'Senior UX Researcher',
  'years_experience_required': 5},
 'cost_usd': 0.000112,
 'latency_s': 2.54,
 'accuracy': 2,
 'parse_success': 1}

In [36]:
class JudgeVerdict(BaseModel):
    score: int      = Field(ge=1, le=4)
    reasoning: str     = Field(min_length=20, max_length=500)

async def score_llm_judge(
    snippet_text: str,
    extracted: dict | None,
    gold: dict
) -> JudgeVerdict:
    RUBRIC = """
4 = The response is parsable as JSON, and all three fields are semantically correct and grounded in the job snippet. Minor differences in capitalization, punctuation, or formatting do not count as errors.
3 = The response is parsable as JSON, two of the three fields are semantically correct and grounded, and the remaining field is incorrect or missing. No field contains fabricated or unsupported information.
2 = The response is parsable as JSON, and one of the three fields is semantically correct, or the response contains a fabricated/unsupported field. A response with two correct fields but one fabricated field is therefore scored 2.
1 = None of the three fields are correct, or the response is unparsable.
"""
    resp = await client.chat.completions.parse(
        model="gpt-4o",
        temperature=0.0,
        response_format=JudgeVerdict,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict evaluator of LLM answers.\n\n"
                    f"{RUBRIC}\n\n"
                    "Evaluate the candidate answer against the job description "
                    "and expected answer. Return a score from 1 to 4 and brief "
                    "reasoning explaining the score."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Job description:\n{snippet_text}\n\n"
                    f"Expected answer:\n"
                    f"company: {gold['company']}\n"
                    f"role: {gold['role']}\n"
                    f"years_experience_required: "
                    f"{gold['years_experience_required']}\n\n"
                    f"Candidate answer:\n{extracted}"
                ),
            },
        ],
    )

    return resp.choices[0].message.parsed

In [37]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
#scored = []   # list of result dicts with scoring fields added
snippets_by_id = {
    item["id"]: item["snippet"]
    for item in snippets
}
for ele in results:
    s_id = ele['snippet_id']
    snippet_text = snippets_by_id.get(s_id, "")
    test_judge = await score_llm_judge(snippet_text,ele['parsed_extraction'], golden.get(s_id))
    ele['llm_judge_score'] = test_judge.score
    ele['llm_judge_reasoning'] = test_judge.reasoning
    
print(f'Scored {len(results)} results.')

Scored 40 results.


In [38]:
with open("run_all_MP1_final.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

## Step 5 — Build the comparison table

In [40]:
df = pd.DataFrame(results)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(6)
summary['parse_success'] = (summary['parse_success'] * 100).round(1)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate (%)', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate (%),Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,100.0,3.9,0.001093,2.5565
few_shot,2.9,100.0,3.9,0.000608,2.2580
structured,2.8,100.0,3.9,0.000451,2.2205
zero_shot,2.7,100.0,3.9,0.000370,1.9460


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```

In [43]:
summary.to_markdown("mp1_comparison.md", index=True)